# Electricity buses, the hydrogen bus and foreign buses

Layer 2: [After the solve](../subsystems/after_the_solve.md). Layer 3: [scripts/market](../modules/market.md).

A PyPSA-GB network has three kinds of bus:

- **GB electrical buses**: the transmission nodes (carrier `AC`, or `DC` for an offshore DC node), where demand,
  generators and storage connect;
- **the hydrogen bus** (carrier `H2`), joined to electrical buses by electrolyser and hydrogen-turbine links, and
  carrying the hydrogen offtake as a Load in MWh of hydrogen;
- **foreign buses**, one at the far end of each interconnector, named with `External` and carrying the foreign
  market as generators and loads.

Every demand total, demand-weighted price and bill counts only the first kind. The rule is one function,
`scripts.market.market_accounting._is_gb_electrical_bus`, and `tests/unit/test_electricity_buses.py` pins it.

In [1]:
import logging, sys, warnings
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Snakefile").exists())
for path in (ROOT, ROOT / "config"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
warnings.filterwarnings("ignore")
logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)
logging.getLogger("scripts").setLevel(logging.WARNING)

import pandas as pd
pd.set_option("display.width", 120)

In [2]:
import pypsa
from scripts.market import market_accounting as ma

snapshots = pd.date_range("2035-01-01", periods=3, freq="h")
n = pypsa.Network()
n.set_snapshots(snapshots)
n.add("Bus", "north", carrier="AC", country="GB")
n.add("Bus", "south", carrier="AC", country="GB")
n.add("Bus", "offshore hub", carrier="DC", country="GB")
n.add("Bus", "H2", carrier="H2", country="")
n.add("Bus", "FR External", carrier="AC", country="FR")

n.add("Load", "north load", bus="north", p_set=pd.Series([100.0, 110.0, 120.0], index=snapshots))
n.add("Load", "south load", bus="south", p_set=50.0)
n.add("Load", "H2 offtake", bus="H2", p_set=30.0)
n.add("Load", "FR demand", bus="FR External", p_set=1000.0)
n.add("Link", "north electrolysis", bus0="north", bus1="H2", p_nom=40.0, efficiency=0.7)
n.add("Link", "IFA", bus0="south", bus1="FR External", p_nom=2000.0)

buses = n.buses[["carrier", "country"]].copy()
buses["GB electrical"] = ma._is_gb_electrical_bus(n.buses)
buses

,carrier,country,GB electrical
name,,,
north,AC,GB,True
south,AC,GB,True
offshore hub,DC,GB,True
H2,H2,,False
FR External,AC,FR,False


The rule reads two columns and the name:

1. the carrier, trimmed and lower-cased, must be `ac`, `dc` or empty (`ELECTRICAL_BUS_CARRIERS`);
2. the name must not contain `External`;
3. where there is a `country` column, it must be `GB` or empty.

An empty carrier counts as electrical because older networks leave it blank; an empty country counts as GB because
only the hydrogen bus is blank, and its carrier already excludes it.

In [3]:
print(sorted(ma.ELECTRICAL_BUS_CARRIERS))
edge_cases = pd.DataFrame({"carrier": [" Ac ", None, "heat", "AC"], "country": ["GB", None, "GB", ""]},
                          index=["spaced", "blank", "heat bus", "IE External"])
edge_cases.assign(**{"GB electrical": ma._is_gb_electrical_bus(edge_cases)})

['', 'ac', 'dc']


,carrier,country,GB electrical
spaced,Ac,GB,True
blank,None,None,True
heat bus,heat,GB,False
IE External,AC,,False


## GB electricity demand

`_gb_electricity_demand_mwh` applies the rule: loads on GB electrical buses, plus consumer flexibility on them, plus
the electricity that conversion links (electrolysers) draw from a GB electrical bus into a non-electrical one. The
hydrogen offtake is MWh of hydrogen, not electricity, and the French load is France's.

In [4]:
n.links_t.p0 = pd.DataFrame({"north electrolysis": [40.0, 0.0, 20.0], "IFA": [500.0, -300.0, 0.0]},
                            index=snapshots)
hours = n.snapshot_weightings.generators
gb = ma._gb_electricity_demand_mwh(n, hours)
every_load = n.loads_t.p_set.reindex(columns=n.loads.index).fillna(n.loads.p_set).sum().sum()
print(f"GB electricity demand: {gb:,.0f} MWh")
print(f"every Load in the network: {every_load:,.0f} MWh")

GB electricity demand: 540 MWh
every Load in the network: 3,570 MWh


330 MWh in the north and 150 MWh in the south, plus 60 MWh into the electrolyser: 540 MWh. Summing every Load
instead would add 90 MWh of hydrogen and 3,000 MWh of French demand.

The bill reads the same rule from a saved network file, without loading it into PyPSA
(`scripts.consumer_bill.project_future_bill._electrical_buses`):

In [5]:
import tempfile
import xarray as xr
from scripts.consumer_bill.project_future_bill import _electrical_buses

with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / "network.nc"
    n.export_to_netcdf(path)
    with xr.open_dataset(path) as ds:
        print(sorted(_electrical_buses(ds)))

['north', 'offshore hub', 'south']


**When you add a bus**, give it a carrier that says what it carries. A new GB electrical node needs `AC` or `DC`; a
new foreign node needs `External` in its name (the interconnector build names them so, see
`scripts/interconnectors/map_to_buses.py`); a bus for another energy carrier needs that carrier. A bus that breaks
these conventions moves demand in or out of every total the market and the bill report.